# H2-Speicher: ESL / ASL aus PyPSA-DE (Phase 0/1)

Auswertung der gelösten Netze je Stützjahr:
- Phase-0-Checks: Zeitauflösung, Kavernen vs. Tanks, Bilanzschluss
- Phase-1-Kennzahlen: ASL, ESL, ESL/ASL, Volllaststunden, AGV, Zyklen
- Energiebilanz der deutschen H2-Busse (Zerlegung von R(t)), Dauerlinien, Spitzenzeitpunkte

Einheiten: Modell in Heizwert (Hi). NEP in Brennwert: Hs = 1,189 × Hi.
Vorzeichen Speicher: > 0 = Ausspeicherung (ASL), < 0 = Einspeicherung (ESL).
Die Rechenlogik liegt in `h2_storage_check.py` im selben Ordner.

In [ ]:
%pip install -q pypsa

In [ ]:
# ==================== PRO LAUF ANPASSEN ====================
RESULTS_DIR = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/KN2045_Mix/results"
OUT_DIR = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/KN2045_Mix/h2_check"
YEARS = [2025, 2030, 2035, 2040, 2045, 2050]
PATTERN = "base_s_27__none_{year}.nc"

# Ordner, in dem h2_storage_check.py liegt
SCRIPT_DIR = "/Workspace/Repos/luisa.kamp@oge.net/pypsa-de-test/analysis/h2_storage"

In [ ]:
import sys
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if SCRIPT_DIR not in sys.path:
    sys.path.append(SCRIPT_DIR)
import h2_storage_check as hc

importlib.reload(hc)  # nach git pull ohne Kernel-Neustart aktuelle Version laden
HS = hc.HS_PER_HI

from IPython.display import HTML
from IPython.display import display as ipy_display


def show(df, fmt=None):
    # Tabelle als HTML anzeigen. Das display() von Databricks wandelt pandas in Spark um,
    # scheitert an gemischten Typen und verwirft den Zeilenindex.
    if fmt is not None:
        html = df.style.format(fmt, na_rep="–").to_html()
    else:
        html = df.to_html(na_rep="–")
    ipy_display(HTML(html))

## 1. Netze auswerten

In [ ]:
summary, balances = hc.run(RESULTS_DIR, years=YEARS, out_dir=OUT_DIR, pattern=PATTERN)

## 2. Phase-0-Checks

- `n_snapshots`: 8760 = stündlich, 2920 = 3H, 24 = 365h (dann Spitzen nicht belastbar)
- `balance_residual_max_abs_MW` sollte ≈ 0 sein, sonst fehlt eine Komponente in der Bilanz
- `store_charger_links` = 0 heißt: keine Leistungsbegrenzung oder -kosten für Ein-/Ausspeicherung

In [ ]:
phase0 = [
    "n_snapshots", "weighting_unique_h", "sum_weights_h",
    "config_clustering_temporal", "config_snapshots", "config_scenario",
    "n_DE_H2_buses", "cavern_buses", "tank_buses",
    "cavern_e_nom_opt_TWh", "cavern_e_nom_max_TWh", "tank_e_nom_opt_TWh",
    "cavern_capital_cost_EUR_per_MWh_a", "tank_capital_cost_EUR_per_MWh_a",
    "store_charger_links", "balance_residual_max_abs_MW",
    "electrolysis_GWel", "h2_power_carriers", "h2_power_GWel", "h2_power_GW_H2in",
]
show(summary.loc[[k for k in phase0 if k in summary.index]])

## 3. Kennzahlen ESL / ASL je Jahr

In [ ]:
def kpi_table(summary, prefix="all_"):
    s = summary.T
    t = pd.DataFrame(index=s.index)
    for lab in ["snap", "day"]:
        t[f"ASL max {lab} [GW_Hi]"] = s[f"{prefix}ASL_max_{lab}"].astype(float) / 1e3
        t[f"ESL max {lab} [GW_Hi]"] = s[f"{prefix}ESL_max_{lab}"].astype(float) / 1e3
        t[f"ESL/ASL max {lab}"] = s[f"{prefix}ESL/ASL_max_{lab}"].astype(float)
        t[f"ASL q99.5 {lab} [GW_Hi]"] = s[f"{prefix}ASL_q995_{lab}"].astype(float) / 1e3
        t[f"ESL q99.5 {lab} [GW_Hi]"] = s[f"{prefix}ESL_q995_{lab}"].astype(float) / 1e3
    t["ASL max [GW_Hs]"] = t["ASL max snap [GW_Hi]"] * HS
    t["ESL max [GW_Hs]"] = t["ESL max snap [GW_Hi]"] * HS
    t["Volllaststunden ASL [h]"] = s[f"{prefix}FLH_ASL_h"].astype(float)
    t["Volllaststunden ESL [h]"] = s[f"{prefix}FLH_ESL_h"].astype(float)
    t["Ausspeicherung [TWh_Hi]"] = s[f"{prefix}discharge_TWh"].astype(float)
    if prefix == "all_":
        t["AGV genutzt [TWh_Hi]"] = s["AGV_used_TWh"].astype(float)
        t["AGV genutzt [TWh_Hs]"] = t["AGV genutzt [TWh_Hi]"] * HS
        t["Zyklen"] = s["cycles"].astype(float)
    return t

kpi = kpi_table(summary)
fmt = {c: ("{:.0%}" if "ESL/ASL" in c else "{:.1f}") for c in kpi.columns}
show(kpi, fmt)

In [ ]:
# 2037 (NEP-Stützjahr) linear aus 2035 und 2040
if {2035, 2040} <= set(kpi.index):
    k2037 = kpi.loc[2035] + (kpi.loc[2040] - kpi.loc[2035]) * 2 / 5
    k2037["ESL/ASL max snap"] = k2037["ESL max snap [GW_Hi]"] / k2037["ASL max snap [GW_Hi]"]
    k2037["ESL/ASL max day"] = k2037["ESL max day [GW_Hi]"] / k2037["ASL max day [GW_Hi]"]
    show(k2037.to_frame("2037 (interpoliert)").T, fmt)

## 4. Jahresbilanz der deutschen H2-Busse [TWh_Hi]

In [ ]:
annual = summary.loc[[k for k in summary.index if k.startswith("annual_TWh")]].astype(float)
annual.index = annual.index.str.replace("annual_TWh[", "", regex=False).str.rstrip("]")
show(annual.round(1))

## 5. Dauerlinien Speicher-Dispatch (Deutschland, alle H2-Speicher)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for y, inj in balances.items():
    store_cols = [c for c in inj.columns if c.startswith("H2 Store")]
    p = inj[store_cols].sum(axis=1) / 1e3
    w = inj.index.to_series().diff().dt.total_seconds().div(3600).bfill()
    s = p.sort_values(ascending=False)
    hours = w.loc[s.index].cumsum()
    ax.plot(hours.values, s.values, label=str(y))
ax.axhline(0, color="grey", lw=0.8)
ax.set_xlabel("Stunden im Jahr")
ax.set_ylabel("GW_Hi  (> 0 Ausspeicherung, < 0 Einspeicherung)")
ax.set_title("Dauerlinie H2-Speicher Deutschland")
ax.legend()
plt.show()

## 6. Zerlegung von R(t) in den Spitzenzeitpunkten

Beiträge aller Komponenten zur Bilanz der deutschen H2-Busse im Zeitschritt mit maximaler
Ausspeicherung (ASL) und maximaler Einspeicherung (ESL). Positiv = Einspeisung in den H2-Bus.

In [ ]:
rows = {}
for y, inj in balances.items():
    store_cols = [c for c in inj.columns if c.startswith("H2 Store")]
    p = inj[store_cols].sum(axis=1)
    for lab, t in {"ASL-Spitze": p.idxmax(), "ESL-Spitze": p.idxmin()}.items():
        r = inj.loc[t].drop("balance residual") / 1e3
        r.name = (y, lab, str(t))
        rows[r.name] = r
peaks = pd.DataFrame(rows).fillna(0.0)
peaks = peaks.loc[peaks.abs().max(axis=1) > 0.05]
show(peaks.round(1))

## 7. Zeitreihe eines Jahres

In [ ]:
YEAR = 2045
inj = balances[YEAR] / 1e3
daily = inj.drop(columns="balance residual").resample("D").mean()
daily = daily.loc[:, daily.abs().max() > 0.1]
fig, ax = plt.subplots(figsize=(14, 6))
daily.plot(ax=ax, lw=1)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("GW_Hi (Tagesmittel, > 0 = Einspeisung in H2-Bus)")
ax.set_title(f"H2-Bilanz Deutschland {YEAR}")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

store_cols = [c for c in balances[YEAR].columns if c.startswith("H2 Store")]
w = balances[YEAR].index.to_series().diff().dt.total_seconds().div(3600).bfill()
soc = (-(balances[YEAR][store_cols].sum(axis=1) * w).cumsum()) / 1e6
fig, ax = plt.subplots(figsize=(14, 3))
(soc - soc.min()).plot(ax=ax)
ax.set_ylabel("TWh_Hi")
ax.set_title(f"Füllstand H2-Speicher Deutschland {YEAR} (relativ zum Minimum)")
plt.show()